# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides an example for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library. The Croissant schema enables structured, FAIR metadata and dataset access.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\nIdentifier: {metadata.identifier}\n")

## 2. Data Overview

Review available record sets, fields, and their IDs. Each entity should be referenced by its `@id`.

Let's enumerate all record sets in the dataset and their fields with associated `@id`s.

In [ ]:
# Enumerate record sets
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

record_sets_ids = []
for rs in record_sets:
    print(f"Record Set: {rs.name} (ID: {rs.id})")
    record_sets_ids.append(rs.id)
    # List associated fields
    if hasattr(rs, 'fields'):
        for fld in rs.fields:
            print(f"  Field: {fld.name} (ID: {fld.id}, DataType: {fld.data_type})")
    print("")

## 3. Data Extraction

Load data from all available record sets into DataFrames, referencing them by `@id`.

You can use the above cell's output to pick a specific record set and its field IDs for further analysis.

In [ ]:
# Load all record sets into dictionaries of DataFrames
dataframes = {}

# If no record sets are present, handle gracefully
if record_sets:
    for rs in record_sets:
        # Collect records for this record set
        try:
            records = list(dataset.records(record_set=rs.id))
            if records:
                df = pd.DataFrame(records)
                dataframes[rs.id] = df
            else:
                print(f"No records found for record set: {rs.id}")
        except Exception as e:
            print(f"Error loading records for record set {rs.id}: {e}")
    # Print column names of the first record set (as example)
    first_rs_id = record_sets_ids[0]
    print(f"\nColumns in record set '{first_rs_id}':")
    if first_rs_id in dataframes:
        print(dataframes[first_rs_id].columns.tolist())
        display(dataframes[first_rs_id].head())
    else:
        print(f"No DataFrame loaded for record set {first_rs_id}")
else:
    print("No record sets found in this dataset schema.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records, normalizing numeric fields, or grouping data, to prepare for further analysis.

We'll select a numeric field from the record set for demonstration (e.g., `gad7_score`, if present) and explore it.

In [ ]:
# EDA on a numeric field from the first record set
# Update these variable values based on the overview above or your dataset's fields
import numpy as np

# Replace with the actual record set and field @id as needed
if record_sets_ids:
    rs_id = record_sets_ids[0]
    df = dataframes[rs_id]

    # Try some common numeric field names
    possible_numeric_fields = ['gad7_score', 'phq9_score', 'psq_score', 'age', 'GAD7', 'PHQ9', 'PSQ']
    numeric_field = None
    for fname in possible_numeric_fields:
        if fname in df.columns:
            numeric_field = fname
            break
    if numeric_field:
        # Filter examples: scores > 10
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (total: {len(filtered_df)}):")
        display(filtered_df[[numeric_field]].head())

        # Normalize the numeric field
        filtered_df[numeric_field + '_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Group by a categorical field if present
        possible_group_fields = ['gender', 'sex', 'marital_status', 'village', 'group']
        group_field = None
        for g in possible_group_fields:
            if g in df.columns:
                group_field = g
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped records by {group_field} (mean {numeric_field}):")
            display(grouped_df)
        else:
            print("No groupable categorical field found for grouping analysis.")
    else:
        print("No obvious numeric field found for EDA in this record set.")
else:
    print("No record sets available for analysis.")

## 5. Visualization

Let's plot the distribution of the selected numeric score field (if available), or show a simple count plot for a categorical field.

This helps quickly review patterns and potential outliers in the survey data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: plot histogram for the numeric field
if record_sets_ids and numeric_field is not None and numeric_field in df:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Optional: boxplot by group field
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable numeric or grouping field for plotting.")

## 6. Conclusion

In this notebook, we've demonstrated how to use the `mlcroissant` library to load, inspect, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. The schema-driven approach enabled structured exploration of record sets and fields, selection by `@id`, and exploratory analysis on key indicators such as psychological assessment scores.

Further analysis (e.g., machine learning, geospatial mapping, multivariate statistics) can be performed on this AI-ready data. For more, see the [FAIR² Croissant schema documentation](https://mlcommons.org/croissant/spec/).
